# โครงงานปลายภาค: CASMI26 PubChem Popularity Prior

**สมาชิก:** นายธีรภัทร พิกุลศรี (รหัสนักศึกษา 68114540308)

**ชุดข้อมูล:** CASMI26 PubChem Popularity Prior จาก Kaggle

โครงงานนี้ศึกษาความสัมพันธ์ระหว่างข้อมูลการอ้างอิงสารเคมีกับคะแนนความนิยม และสร้างแบบจำลอง regression เพื่อทำนายคะแนนดังกล่าว


## Part 1: ภาพรวมข้อมูลและ EDA

ตรวจสอบขนาดข้อมูล ชนิดข้อมูล ค่าที่หายไป สถิติเชิงพรรณนา และความสัมพันธ์ระหว่างตัวแปร


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

DATA_DIR = Path(kagglehub.dataset_download('dmitriigluzdov/casmi26-pubchem-popularity-prior'))
if not (DATA_DIR / 'pool_lsid.npy').exists():
    DATA_DIR = next(DATA_DIR.glob('versions/*'))
load = lambda name: np.load(DATA_DIR / f'{name}.npy', mmap_mode='r')
lsid = np.asarray(load('pool_lsid'), dtype=np.float32)
lpmid = np.asarray(load('pool_lpmid'), dtype=np.float32)
lpatent = np.asarray(load('pool_lpatent'), dtype=np.float32)
cid = np.asarray(load('pool_cid'), dtype=np.int32)
np_bits = np.asarray(load('pool_np'), dtype=np.uint8)
df = pd.DataFrame({'log_sid': lsid, 'log_pmid': lpmid,
                   'has_pubchem': (cid > 0).astype(np.int8),
                   'is_natural_product': (np_bits > 0).astype(np.int8),
                   'target_popularity': lsid + lpmid + lpatent})
df = df.sample(n=100000, random_state=42).reset_index(drop=True)
feature_cols = ['log_sid', 'log_pmid', 'has_pubchem', 'is_natural_product']
X = df[feature_cols].to_numpy(float)
y = df['target_popularity'].to_numpy(float)
print('ขนาดข้อมูล:', df.shape)
print('ค่าที่หายไป:\n', df.isna().sum())
display(df.describe())


## Part 2: CLO1 — พีชคณิตเชิงเส้น

คำนวณ covariance matrix, eigendecomposition และ PCA เพื่อดูทิศทางความแปรปรวนหลักของข้อมูล


In [ ]:
X_scaled = StandardScaler().fit_transform(X)
covariance = np.cov(X_scaled, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]
explained = eigenvalues / eigenvalues.sum()
X_pca = X_scaled @ eigenvectors[:, :2]
print('สัดส่วนความแปรปรวนที่อธิบายได้ (%):', np.round(explained * 100, 2))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, s=5, alpha=.25, cmap='viridis')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('การฉายข้อมูลด้วย PCA'); plt.colorbar(label='คะแนนความนิยม'); plt.show()


## Part 3: CLO2 — การเรียนรู้เชิงสถิติ

ตรวจสอบความสัมพันธ์กับ target และแสดง bias-variance trade-off ด้วย polynomial regression


In [ ]:
print('ความสัมพันธ์กับ target:')
print(df[feature_cols + ['target_popularity']].corr()['target_popularity'].sort_values(ascending=False))
x_one = df[['log_sid']].to_numpy()
x_train, x_test, y_train, y_test = train_test_split(x_one, y, test_size=.2, random_state=42)
train_mse, test_mse = [], []
for degree in range(1, 8):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x_train, y_train)
    train_mse.append(mean_squared_error(y_train, model.predict(x_train)))
    test_mse.append(mean_squared_error(y_test, model.predict(x_test)))
plt.plot(range(1, 8), train_mse, 'o-', label='Train MSE')
plt.plot(range(1, 8), test_mse, 'o-', label='Test MSE')
plt.xlabel('ระดับ polynomial'); plt.ylabel('MSE'); plt.title('ความสัมพันธ์ระหว่าง bias และ variance'); plt.legend(); plt.show()


## Part 4: CLO3 — การสร้างแบบจำลอง

เปรียบเทียบ Simple Linear Regression กับ Ridge Regression ด้วย MSE และ R²


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
simple = LinearRegression().fit(X_train[:, [0]], y_train)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_train, y_train)
for name, actual, pred in [('Simple Linear Regression', y_test, simple.predict(X_test[:, [0]])), ('Ridge', y_test, ridge.predict(X_test))]:
    print(name, 'MSE =', round(mean_squared_error(actual, pred), 4), 'R2 =', round(r2_score(actual, pred), 4))


## Part 5: CLO4 — การเลือกโมเดล

ใช้ 5-fold cross-validation เพื่อเลือกโมเดลที่มีความสามารถทั่วไปดีที่สุด


In [ ]:
models = {
    'Simple Linear': (make_pipeline(StandardScaler(), LinearRegression()), X[:, [0]]),
    'Ridge': (make_pipeline(StandardScaler(), Ridge(alpha=1.0)), X),
    'Polynomial degree 2': (make_pipeline(PolynomialFeatures(2), Ridge(alpha=1.0)), X[:, [0]])
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []
for name, (model, features) in models.items():
    scores = -cross_val_score(model, features, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    results.append((name, scores.mean(), scores.std()))
results = pd.DataFrame(results, columns=['Model', 'CV MSE mean', 'CV MSE std']).sort_values('CV MSE mean')
display(results)
print('โมเดลที่ดีที่สุด:', results.iloc[0]['Model'])


## Part 6: สรุปผลและเรื่องราวจากข้อมูล

- PCA แสดงทิศทางความแปรปรวนหลักของข้อมูล
- `log_sid` และ `log_pmid` มีความสัมพันธ์กับ popularity สูง
- Ridge ใช้หลาย features และ regularization จึงเหมาะกว่าโมเดลพื้นฐาน
- Cross-validation ช่วยเลือกโมเดลอย่างเป็นธรรม

**ข้อจำกัด:** target popularity เป็น prior จาก count data ไม่ใช่ค่าคุณภาพของสารโดยตรง และใช้ sample 100,000 แถวเพื่อให้รันได้บนคอมพิวเตอร์ทั่วไป
